# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR² Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We examine the record sets described in the Croissant schema, their `@id`s, and the field structure for data exploration.

> **Note**: All record sets, fields, and columns are referenced by their `@id` as per Croissant schema.

In [ ]:
# List available record sets and their @id, as well as fields in each record set
record_sets = metadata.record_sets  # This returns a list of RecordSet objects
print(f"Number of record sets in this dataset: {len(record_sets)}\n")

for record_set in record_sets:
    print(f"Record set name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print("  Fields (columns):")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print()

## 3. Data Extraction
Load data from an available record set into a DataFrame for analysis. 
Below, we use the record set and field `@id`s from the overview above. For demonstration, we will use the main record set containing the protein information.

In [ ]:
# List all record set IDs
all_record_set_ids = [rs.id for rs in metadata.record_sets]
print('All available record set @id values:')
pprint.pprint(all_record_set_ids)

# Assign the main data record set (assumed to be the first record set here)
main_record_set_id = all_record_set_ids[0]

# Extract data from each record set and store as DataFrames
dataframes = {}
for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from record set '@id': {record_set_id}")

# Print columns of the main record set
print(f"\nColumn names in main record set '{main_record_set_id}':")
print(dataframes[main_record_set_id].columns.tolist())

# Show a preview
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

For demonstration, let's assume there is a numeric field representing peptide count or protein abundance in the main record set. We'll reference all fields by schema `@id` where possible.

In [ ]:
# Identify a numeric field to analyze by @id
# You may need to adjust the variable below to select an available numeric field in your dataset.
main_fields = {f.name: f.id for f in metadata.get_record_set(main_record_set_id).fields}
print('Available fields and their @id:')
pprint.pprint(main_fields)

# Examples: use peptide counts, abundance, or MW, depending on what's available
# Select a field, e.g., 'Peptide count' or 'Abundance', by examining the printed fields
numeric_field_id = None
for field in metadata.get_record_set(main_record_set_id).fields:
    # A heuristic to find a likely numeric field
    if ('peptide' in field.name.lower() or 'abundance' in field.name.lower() or 'count' in field.name.lower() or 'mw' in field.name.lower() or 'weight' in field.name.lower()) and ('string' not in field.data_type):
        numeric_field_id = field.id
        break

if numeric_field_id is None:
    # fallback to first field (for demonstration only)
    numeric_field_id = list(main_fields.values())[0]

print(f"Selected numeric field: {numeric_field_id}")

# Filtering rows where value > threshold
df_main = dataframes[main_record_set_id]
# Try to convert to numeric (in case data came as string)
df_main[numeric_field_id] = pd.to_numeric(df_main[numeric_field_id], errors='coerce')
threshold = df_main[numeric_field_id].dropna().quantile(0.9)
filtered_df = df_main[df_main[numeric_field_id] > threshold]

print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (90th percentile):")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].copy()].head())

# Group by a likely categorical field, e.g., 'description', 'sample', or similar
group_field_id = None
for field in metadata.get_record_set(main_record_set_id).fields:
    if ('desc' in field.name.lower() or 'sample' in field.name.lower() or 'accession' in field.name.lower()):
        group_field_id = field.id
        break

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we show a histogram of the selected numeric field and a boxplot by group if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the (possibly normalized) numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df_main[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouping is available, show boxplot
if group_field_id and group_field_id in df_main.columns:
    plt.figure(figsize=(10, 5))
    # Only plot top 10 frequent group values
    top_group_values = df_main[group_field_id].value_counts().index[:10]
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df_main[df_main[group_field_id].isin(top_group_values)])
    plt.title(f"{numeric_field_id} by {group_field_id} (Top 10 groups)")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load and inspect the FAIR² dataset: *Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells*.

- We loaded and previewed the dataset metadata and record sets via Croissant schema using their `@id`s.
- We extracted and explored fields by `@id`, loaded tabular data as DataFrames, filtered using numeric features, and normalized them.
- We visualized distribution of a key numeric field and examined its grouping by categorical attributes.

For further analysis, consult the [FAIR² dataset documentation](https://sen.science/doi/10.71728/senscience.vq4a-28xa) and the [mlcroissant documentation](https://mlcommons.github.io/croissant/) for more advanced workflows.